# Fast out-of-sample GARCH evaluation

This notebook fits each model **once per asset** on the training sample. It then keeps the fitted parameters fixed and generates rolling one-step-ahead forecasts conditioned on the returns observed up to each forecast date.

This replaces the original expensive approach, which refitted the model for every test observation.


In [ ]:
from pathlib import Path
from time import perf_counter
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)

TRAIN_PATH = Path("/mnt/data/2563195e-5d7f-4a48-816c-c5422d51d5a9.parquet")
TEST_PATH = Path("/mnt/data/ae612d75-c2af-45cb-b63b-47df320c0a53.parquet")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use 1 while debugging. Use -1 to use all CPU cores.
N_JOBS = -1


In [ ]:
train_returns = pd.read_parquet(TRAIN_PATH).sort_index()
test_returns = pd.read_parquet(TEST_PATH).sort_index()

# Keep only numeric columns present in both datasets.
asset_cols = [
    c for c in train_returns.columns
    if c in test_returns.columns
    and pd.api.types.is_numeric_dtype(train_returns[c])
    and pd.api.types.is_numeric_dtype(test_returns[c])
]

if not asset_cols:
    raise ValueError("No common numeric asset columns were found.")

if train_returns.index.has_duplicates or test_returns.index.has_duplicates:
    raise ValueError("Train/test indices contain duplicate timestamps.")

if len(train_returns.index.intersection(test_returns.index)):
    raise ValueError("Train and test indices overlap. Use a strictly out-of-sample split.")

print(f"Train shape: {train_returns.shape}")
print(f"Test shape:  {test_returns.shape}")
print(f"Assets:      {len(asset_cols)}")


In [ ]:
MODEL_SPECS = {
    "GARCH(1,1)": {"vol": "GARCH", "p": 1, "o": 0, "q": 1},
    "GJR-GARCH(1,1)": {"vol": "GARCH", "p": 1, "o": 1, "q": 1},
    "EGARCH(1,1)": {"vol": "EGARCH", "p": 1, "o": 1, "q": 1},
}

RETURN_SCALE = 100.0  # improves numerical conditioning for decimal returns
MIN_TRAIN_OBS = 100


def fit_and_forecast_one(asset, model_name, spec):
    """Fit once on train and make rolling 1-step forecasts using fixed parameters."""
    train = train_returns[asset].dropna().astype(float)
    test = test_returns[asset].dropna().astype(float)

    if len(train) < MIN_TRAIN_OBS or test.empty:
        return None

    # Remove non-finite values explicitly.
    train = train[np.isfinite(train)]
    test = test[np.isfinite(test)]
    if len(train) < MIN_TRAIN_OBS or test.empty:
        return None

    y_train = train * RETURN_SCALE
    y_all = pd.concat([train, test]) * RETURN_SCALE

    # Fit only on training data.
    train_model = arch_model(
        y_train,
        mean="Constant",
        dist="t",
        rescale=False,
        **spec,
    )
    fit = train_model.fit(disp="off", show_warning=False, update_freq=0)

    # Rebuild on train+test and fix training-estimated parameters.
    # Forecasts can then use realized lagged test returns without re-estimation.
    full_model = arch_model(
        y_all,
        mean="Constant",
        dist="t",
        rescale=False,
        **spec,
    )
    fixed = full_model.fix(fit.params)

    forecasts = fixed.forecast(
        horizon=1,
        start=len(train) - 1,
        align="target",
        reindex=True,
    )

    # Target-aligned h.1 forecast at each test timestamp.
    variance_scaled = forecasts.variance["h.1"].reindex(test.index)
    variance = variance_scaled / (RETURN_SCALE ** 2)
    variance = variance.clip(lower=0)

    realized_variance = test.pow(2)
    valid = variance.notna() & realized_variance.notna()
    variance = variance[valid]
    realized_variance = realized_variance[valid]

    if variance.empty:
        return None

    errors = variance - realized_variance
    rmse = float(np.sqrt(np.mean(errors.pow(2))))
    mae = float(np.mean(errors.abs()))

    # QLIKE is often more informative for volatility models; included as a bonus.
    eps = np.finfo(float).eps
    qlike = float(np.mean(np.log(variance.clip(lower=eps)) + realized_variance / variance.clip(lower=eps)))

    forecast_table = pd.DataFrame({
        "asset": asset,
        "model": model_name,
        "return": test.reindex(variance.index),
        "realized_variance": realized_variance,
        "forecast_variance": variance,
        "forecast_volatility": np.sqrt(variance),
    })

    metric_row = {
        "asset": asset,
        "model": model_name,
        "n_train": len(train),
        "n_test": len(variance),
        "converged": fit.convergence_flag == 0,
        "loglikelihood": float(fit.loglikelihood),
        "rmse": rmse,
        "mae": mae,
        "qlike": qlike,
    }
    return forecast_table, metric_row


In [ ]:
jobs = [
    (asset, model_name, spec)
    for asset in asset_cols
    for model_name, spec in MODEL_SPECS.items()
]

start = perf_counter()
results = Parallel(n_jobs=N_JOBS, prefer="processes")(
    delayed(fit_and_forecast_one)(asset, model_name, spec)
    for asset, model_name, spec in jobs
)
results = [r for r in results if r is not None]
elapsed = perf_counter() - start

if not results:
    raise RuntimeError("No models produced valid forecasts.")

forecast_long = pd.concat([r[0] for r in results]).sort_index()
metrics = pd.DataFrame([r[1] for r in results]).sort_values(["asset", "rmse"])

forecast_long.to_parquet(OUTPUT_DIR / "garch_test_forecasts.parquet")
metrics.to_csv(OUTPUT_DIR / "garch_test_metrics.csv", index=False)

print(f"Completed {len(results)} asset-model fits in {elapsed:,.2f} seconds.")
display(metrics)


In [ ]:
# Best model for each asset according to RMSE and MAE
best_rmse = metrics.loc[metrics.groupby("asset")["rmse"].idxmin()].sort_values("asset")
best_mae = metrics.loc[metrics.groupby("asset")["mae"].idxmin()].sort_values("asset")

print("Best by RMSE")
display(best_rmse[["asset", "model", "rmse", "mae", "qlike", "converged"]])

print("Best by MAE")
display(best_mae[["asset", "model", "rmse", "mae", "qlike", "converged"]])


In [ ]:
# Optional diagnostic plot for one asset
asset_to_plot = asset_cols[0]
plot_data = forecast_long[forecast_long["asset"] == asset_to_plot]

plt.figure(figsize=(14, 5))
plt.plot(
    plot_data.index.unique(),
    test_returns.loc[plot_data.index.unique(), asset_to_plot].abs(),
    label="Absolute return",
    linewidth=1,
)
for model_name, group in plot_data.groupby("model"):
    plt.plot(group.index, group["forecast_volatility"], label=model_name, linewidth=1.2)
plt.title(f"{asset_to_plot}: one-step-ahead volatility forecasts")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Why this is faster

- Original complexity: approximately `assets × models × test observations` numerical optimizations.
- Revised complexity: only `assets × models` numerical optimizations.
- Test observations update the conditional variance through fixed fitted parameters, rather than triggering a new maximum-likelihood fit.
- Assets/models can run in parallel with `joblib`.

## Evaluation note

A latent conditional variance is not directly observable. The notebook uses squared returns as the realized-variance proxy, so RMSE and MAE compare `forecast variance` against `return²`. For financial volatility forecasts, QLIKE is also reported because it is commonly more robust to noisy variance proxies.
